In [1]:
from pathlib import Path
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [2]:
load_dotenv()

OPENAI_LLM_MODEL = "gpt-4.1-mini"
OPENAI_EMBEDDING_MODEL = "text-embedding-3-small"

BASE_DIR = Path(r"C:\Users\Playdata\workspace\SKN28-third-2TEAM")

CHROMA_DIR = BASE_DIR / "vectorstore" / "kaist_ai_grad_chroma"
COLLECTION_NAME = "kaist_ai_graduate_rag"

print(CHROMA_DIR)

C:\Users\Playdata\workspace\SKN28-third-2TEAM\vectorstore\kaist_ai_grad_chroma


In [3]:
embeddings = OpenAIEmbeddings(
    model=OPENAI_EMBEDDING_MODEL
)

vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=str(CHROMA_DIR),
)

print("Vector Store 로드 완료")

Vector Store 로드 완료


In [4]:
retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 4
    }
)

In [5]:
query = "AI시스템학과 입학 정보를 알려줘"

docs = retriever.invoke(query)

for i, doc in enumerate(docs, 1):
    print(f"\n===== 문서 {i} =====")
    print("metadata:", doc.metadata)
    print("content:", doc.page_content[:500])


===== 문서 1 =====
metadata: {'source_url': 'https://ai-systems.kaist.ac.kr/admission-grad', 'dept_name': 'AI시스템학과', 'doc_id': 'doc_08e76f55f487', 'title': 'AI시스템학과 소개서 다운로드 Download AI Systems Brochure', 'dept': 'ai_systems', 'data_origin': 'rag_chunks_csv', 'source_type': 'admission', 'section_path': '대학원과정 입학 안내 Graduate Admission', 'chunk_id': 'chunk_00004'}
content: [학과] AI시스템학과 [문서] AI시스템학과 소개서 다운로드 Download AI Systems Brochure [섹션] 대학원과정 입학 안내 Graduate Admission [유형] admission AI시스템학과 소개서 다운로드 Download AI Systems Brochure 박사과정, 석사과정, (KAIST 석사재학생) 석박사 통합과정, We recruit new students for doctoral, master's, and integrated programs, (KAIST 학사재학생) 학-석박통합연계과정 신입생을 모집합니다. including the undergraduate-to-master's integrated track for KAIST undergraduates. AI시스템학과 소개서 다운로드 Download AI Systems Brochure 입학처 홈페이지 바로가기

===== 문서 2 =====
metadata: {'doc_id': 'doc_97204880a933', 'title': '입학처 홈페이지 바로가기 Go to Admissions Website', 'data_origin': 'rag_chunks_csv', 'source_type': 'asset', 'dept': 'a

In [6]:
system_prompt = """
당신은 KAIST 대학원 정보를 안내하는 RAG 챗봇입니다.

아래 context에 포함된 내용만 근거로 답변하세요.
context에 없는 내용은 추측하지 말고 "제공된 자료에서 확인되지 않습니다."라고 답변하세요.

답변 규칙:
1. 한국어로 답변하세요.
2. 사용자가 이해하기 쉽게 정리하세요.
3. 관련 학과명, 출처, 문서명이 있으면 함께 언급하세요.
4. 모르는 내용은 절대 지어내지 마세요.

<context>
{context}
</context>
"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{question}")
    ]
)

In [7]:
def format_docs(docs):
    formatted = []

    for i, doc in enumerate(docs, 1):
        metadata = doc.metadata

        source = metadata.get("title") or metadata.get("source_file") or metadata.get("source_url") or "출처 없음"
        dept_name = metadata.get("dept_name", "")

        formatted.append(
            f"""
[문서 {i}]
학과: {dept_name}
출처: {source}
내용:
{doc.page_content}
"""
        )

    return "\n\n".join(formatted)

In [8]:
llm = ChatOpenAI(
    model=OPENAI_LLM_MODEL,
    temperature=0
)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [9]:
question = "AI시스템학과는 어떤 학과야?"

answer = rag_chain.invoke(question)

print(answer)

AI시스템학과는 실리콘과 하드웨어부터 알고리즘, 소프트웨어, 응용 시스템까지 AI 전 영역을 아우르는 통합형 AI 시스템 전문가를 양성하는 학과입니다. 

주요 특징은 다음과 같습니다(출처: AI시스템학과 학과소개 Introduction):
- 하드웨어와 소프트웨어의 경계를 허무는 HW/SW 코디자인 교육을 기반으로 AI 시스템과 인프라 기술 전반을 깊이 이해하고 설계할 수 있는 인재를 체계적으로 육성합니다.
- 전기전자공학 분야를 중심으로 AI 반도체, 하드웨어, 컴퓨팅 아키텍처, 소프트웨어, 알고리즘, 물리적 AI 시스템까지 전주기 AI 교육 체계를 갖추고 있습니다.
- 이론, 시스템, 응용을 유기적으로 연결하는 통합 교육을 통해 학생들이 AI 기술을 개별 요소가 아닌 하나의 완성된 시스템으로 바라보고 구현하는 역량을 키웁니다.
- 경계를 넘는 질문과 혁신을 통해 새로운 패러다임을 이끄는 탐험가·창조자·리더형 인재, 즉 미래 AI 시대 핵심 인프라 계층을 담당할 AI 하드웨어와 소프트웨어 통합 스페셜리스트를 양성합니다.

요약하면, AI시스템학과는 AI 하드웨어와 소프트웨어를 통합적으로 설계하고 구현할 수 있는 차세대 AI 시스템 전문가를 키우는 학과입니다.
